# Machine Learning for Phishing URL Detection

**Security problem:** Detect phishing URLs using structural and lexical URL features.

**Research question:** Can machine-learning models accurately identify phishing URLs based on features found in the URL?

**Dataset:** Phishing URLs Dataset with Extracted Features (Kaggle)  
https://www.kaggle.com/datasets/victusadi/phishing-urls-dataset-with-extracted-features


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    ConfusionMatrixDisplay
)

df = pd.read_csv("phishing_features.csv")
df.head()


## Step 1 — Dataset Overview

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nClass distribution:")
print(df["label"].value_counts())
print("\nClass percentages:")
print(df["label"].value_counts(normalize=True) * 100)


## Step 2 — Security Problem
Phishing sites attempt to trick users into sharing credentials or other sensitive information. This project tests whether URL features can help classify phishing and legitimate URLs.

## Step 3 — Exploratory Data Analysis

In [ ]:
print("Missing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

df["label"].value_counts().plot(kind="bar")
plt.title("Class Distribution")
plt.xlabel("Label")
plt.ylabel("Count")
plt.xticks([0, 1], ["Legitimate", "Phishing"], rotation=0)
plt.show()

features = ["url_length","num_dots","num_subdirs","digits_count","entropy"]
print("\nMean feature values by label:")
print(df.groupby("label")[features].mean())

numeric_df = df.select_dtypes(include=["number"])
corr = numeric_df.corr()
plt.figure(figsize=(10,8))
plt.imshow(corr, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## Step 4 — Preprocessing

In [ ]:
data = df.copy()
data["tld"] = data["tld"].fillna("Unknown")

X = data.drop(columns=["label", "url"])
y = data["label"]

categorical_features = ["tld"]
numeric_features = [c for c in X.columns if c != "tld"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor_lr = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

preprocessor_rf = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


## Step 5 — Train Two Machine-Learning Models

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor_lr),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

random_forest_model = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)

logistic_predictions = logistic_model.predict(X_test)
rf_predictions = random_forest_model.predict(X_test)


## Step 6 — Model Evaluation

In [ ]:
def evaluate_model(name, model, predictions):
    probabilities = model.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()

    results = {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
        "False Positive Rate": fp / (fp + tn),
        "False Negative Rate": fn / (fn + tp),
    }

    print(name)
    for key, value in results.items():
        print(f"{key}: {value:.4f}")
    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, predictions))
    print("\nClassification report:")
    print(classification_report(y_test, predictions))

    ConfusionMatrixDisplay.from_predictions(
        y_test, predictions,
        display_labels=["Legitimate", "Phishing"]
    )
    plt.title(f"{name} Confusion Matrix")
    plt.show()

    return results

lr_results = evaluate_model("Logistic Regression", logistic_model, logistic_predictions)
rf_results = evaluate_model("Random Forest", random_forest_model, rf_predictions)


### Observed results from the submitted dataset

Using the 80/20 stratified split and the full engineered feature set, both Logistic Regression and Random Forest produced perfect test-set scores in our run. This result should **not** be interpreted as proof of perfect real-world phishing detection. The dataset has a strong separation artifact: for example, `num_subdirs` is 0 for every legitimate URL in the dataset, while phishing URLs have at least 2. This likely makes the classes unusually easy to separate and is an important dataset limitation.


## Optional Advanced Component — Feature Importance

In [ ]:
feature_names = list(numeric_features)
ohe = random_forest_model.named_steps["preprocessor"].named_transformers_["cat"]
feature_names += list(ohe.get_feature_names_out(["tld"]))

importances = pd.Series(
    random_forest_model.named_steps["classifier"].feature_importances_,
    index=feature_names
).sort_values(ascending=False)

print(importances.head(15))
importances.head(10).sort_values().plot(kind="barh")
plt.title("Random Forest Top Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## Step 7 — Security Interpretation

False positives can cause legitimate websites to be blocked or generate unnecessary analyst work. False negatives are particularly dangerous because a phishing URL may be treated as safe, potentially exposing users to credential theft or fraud.

The model could support a real security system as one risk signal, but it should not be the only control. A production system should combine URL features with domain reputation, page content, certificate data, threat intelligence, and human review.

## Limitations

- The dataset is extremely imbalanced.
- The dataset appears to contain feature/source biases that make class separation unusually easy.
- Results on a random split from this same dataset may not generalize to new phishing campaigns.
- URLs and attacker behavior change over time.
- The project does not inspect webpage content or external reputation signals.

## Conclusion

The project demonstrates how machine learning can be applied to phishing detection, while also showing why evaluation quality and dataset design matter as much as raw accuracy.
